# Notebook 03 — Exploratory Data Analysis (EDA)

**Author:** Section D – Group 8  
**Input:** `data/processed/clean_data.csv`  

**Goal:** Understand the US residential real estate market through descriptive statistics, geographic analysis, price distributions, and correlation patterns.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:.2f}'.format)

df = pd.read_csv('../data/processed/clean_data.csv')
print(f'Dataset shape: {df.shape}')
df.head(3)

In [ ]:
# Column list
print('Columns:', list(df.columns))

In [ ]:
# Quick descriptive stats on key numeric columns
df[['Sale_Price', 'Living_Space_SqFt', 'Bedrooms', 'Bathrooms', 'Price_Per_SqFt', 'land_space']].describe().round(2)

## 1. Price Distribution

In [ ]:
# Distribution of raw sale price
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['Sale_Price'].dropna(), bins=80, color='steelblue', edgecolor='white')
ax.set_title('Sale Price Distribution (Raw)', fontsize=14)
ax.set_xlabel('Sale Price ($)')
ax.set_ylabel('Count')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
plt.tight_layout()
plt.show()

In [ ]:
# Distribution using CAPPED price (removes skew from extreme outliers)
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['Sale_Price_Capped'].dropna(), bins=80, color='teal', edgecolor='white')
ax.set_title('Sale Price Distribution (Capped at 99th Percentile)', fontsize=14)
ax.set_xlabel('Sale Price ($)')
ax.set_ylabel('Count')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
plt.tight_layout()
plt.show()

In [ ]:
# Price percentiles
for pct in [10, 25, 50, 75, 90, 95, 99]:
    val = df['Sale_Price'].quantile(pct/100)
    print(f'{pct:3}th percentile: ${val:>12,.0f}')

## 2. Property Type Breakdown

In [ ]:
# Count and share by property type
pt = df['property_type'].value_counts().reset_index()
pt.columns = ['Property Type', 'Count']
pt['Share (%)'] = (pt['Count'] / pt['Count'].sum() * 100).round(2)
print(pt.to_string(index=False))

In [ ]:
# Pie chart
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(pt['Count'], labels=pt['Property Type'], autopct='%1.1f%%',
       colors=sns.color_palette('Set2', len(pt)))
ax.set_title('Property Type Distribution', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Median price per property type
price_by_type = df.groupby('property_type')['Sale_Price_Capped'].median().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
price_by_type.plot(kind='bar', ax=ax, color=sns.color_palette('viridis', len(price_by_type)))
ax.set_title('Median Sale Price by Property Type', fontsize=14)
ax.set_xlabel('Property Type')
ax.set_ylabel('Median Price ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M' if x >= 1e6 else f'${x/1e3:.0f}K'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 3. Geographic Analysis — State Level

In [ ]:
# Listing count per state
state_counts = df['state'].value_counts().head(20)
fig, ax = plt.subplots(figsize=(12, 5))
state_counts.plot(kind='bar', ax=ax, color='cornflowerblue')
ax.set_title('Top 20 States by Number of Listings', fontsize=14)
ax.set_xlabel('State')
ax.set_ylabel('Listing Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Median price per state (top 20 by count)
top_states = df['state'].value_counts().head(20).index
state_price = df[df['state'].isin(top_states)].groupby('state')['Sale_Price_Capped'].median().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
state_price.plot(kind='bar', ax=ax, color='coral')
ax.set_title('Median Sale Price — Top 20 States', fontsize=14)
ax.set_xlabel('State')
ax.set_ylabel('Median Price ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M' if x >= 1e6 else f'${x/1e3:.0f}K'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# State-level summary table
state_summary = df.groupby('state').agg(
    Listings=('Sale_Price', 'count'),
    Median_Price=('Sale_Price_Capped', 'median'),
    Mean_Price=('Sale_Price_Capped', 'mean'),
    Median_SqFt=('Living_Space_Capped', 'median')
).sort_values('Listings', ascending=False).head(20)
state_summary.round(0)

## 4. Living Space Analysis

In [ ]:
# Living space distribution (capped)
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['Living_Space_Capped'].dropna(), bins=60, color='mediumseagreen', edgecolor='white')
ax.set_title('Living Space Distribution (Capped at 99th Pct)', fontsize=14)
ax.set_xlabel('Living Space (SqFt)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Median living space by bedrooms
bed_space = df.groupby('Bedrooms')['Living_Space_SqFt'].median().loc[lambda x: x.index <= 8]
print('Median Living Space by Bedroom Count:')
print(bed_space.round(0))

## 5. Price Per SqFt Analysis

In [ ]:
# Trim extreme price/sqft outliers for visualization
pps_99 = df['Price_Per_SqFt'].quantile(0.99)
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df.loc[df['Price_Per_SqFt'] <= pps_99, 'Price_Per_SqFt'].dropna(), bins=60, color='mediumpurple', edgecolor='white')
ax.set_title('Price Per SqFt Distribution', fontsize=14)
ax.set_xlabel('$/SqFt')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 most expensive cities by median $/sqft
city_pps = df.groupby('city')['Price_Per_SqFt'].median()
city_pps = city_pps[city_pps.index != 'Unknown City']
print('Top 10 Most Expensive Cities ($/SqFt):')
print(city_pps.sort_values(ascending=False).head(10).round(2))

## 6. Bedroom & Bathroom Distribution

In [ ]:
# Bedroom count distribution
bed_counts = df['Bedrooms'].value_counts().loc[lambda x: x.index <= 8].sort_index()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(bed_counts.index, bed_counts.values, color='steelblue')
axes[0].set_title('Bedroom Count Distribution')
axes[0].set_xlabel('Bedrooms')
axes[0].set_ylabel('Count')

bath_counts = df['Bathrooms'].value_counts().loc[lambda x: x.index <= 6].sort_index()
axes[1].bar(bath_counts.index, bath_counts.values, color='coral')
axes[1].set_title('Bathroom Count Distribution')
axes[1].set_xlabel('Bathrooms')

plt.tight_layout()
plt.show()

## 7. Property Status Distribution

In [ ]:
status_counts = df['property_status'].value_counts()
print('Property Status:')
print(status_counts)

fig, ax = plt.subplots(figsize=(7, 5))
status_counts.plot(kind='bar', ax=ax, color=sns.color_palette('Set1', len(status_counts)))
ax.set_title('Property Status Distribution', fontsize=14)
ax.set_xlabel('Status')
ax.set_ylabel('Count')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 8. Correlation Heatmap

In [ ]:
corr_cols = ['Sale_Price_Capped', 'Living_Space_Capped', 'Bedrooms', 'Bathrooms', 'Price_Per_SqFt', 'land_space']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: Price vs Living Space
sample = df.sample(3000, random_state=1)
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(sample['Living_Space_Capped'], sample['Sale_Price_Capped'],
           alpha=0.3, s=10, color='steelblue')
ax.set_title('Sale Price vs Living Space (3k sample)', fontsize=13)
ax.set_xlabel('Living Space (SqFt)')
ax.set_ylabel('Sale Price ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
plt.tight_layout()
plt.show()

## 9. Price Box Plot by Property Type

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=df, x='property_type', y='Sale_Price_Capped', palette='Set2', ax=ax)
ax.set_title('Price Distribution by Property Type (Capped)', fontsize=14)
ax.set_xlabel('Property Type')
ax.set_ylabel('Sale Price ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M' if x >= 1e6 else f'${x/1e3:.0f}K'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 10. Top Agencies by Listing Count

In [ ]:
top_agencies = df[df['agency_name'] != 'Unknown Agency']['agency_name'].value_counts().head(10)
print('Top 10 Agencies by Listing Count:')
print(top_agencies)